# NurtureNet — Final Training Pipeline
GCAN V4 combines video, audio, speech, on-screen text and metadata to classify videos as **Safe, Harmful Content, Adult Content or Suicide**.

**Run:** install `torch`, `numpy`, `pandas`, `scikit-learn` and `IPython`; set `NURTURENET_ROOT` to the original project folder. Google Drive mounting is supported. Default mode evaluates the saved epoch-12 checkpoint; `RUN_TRAINING=True` starts a new run.

All 21 code cells were executed successfully. V4 results matched the saved artifacts; full retraining and the baseline were not rerun. Source gaps are listed in section 21.


## 2. Imports and reproducibility
Import dependencies and set seed 42. Fresh training may vary across hardware and library versions.


In [1]:
from pathlib import Path
import os
import re
import sys
import json
import random
import time
import hashlib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from IPython.display import display, HTML

SEED = 42
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Device:', device)


Python: 3.11.16 | PyTorch: 2.14.0 | Device: cpu


## 3. Configuration and paths
Centralize paths and settings. Read original assets from `BASE` and save new training results to `RUN_DIR`.


In [2]:
MOUNT_GOOGLE_DRIVE = 'google.colab' in sys.modules
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

candidates = [
    Path('/content/drive/MyDrive/NurtureNet'),
    Path.home() / 'Library/CloudStorage/GoogleDrive-abdulrehmantahir142@gmail.com/My Drive/NurtureNet',
    Path(r'G:\My Drive\NurtureNet'),
    Path.home() / 'Desktop/NurtureNet',
]
NURTURENET_ROOT = os.environ.get('NURTURENET_ROOT', '')  # Set explicitly on another computer.
BASE = Path(NURTURENET_ROOT).expanduser() if NURTURENET_ROOT else next(
    (p for p in candidates if (p / 'processed/training_features/feature_index.csv').is_file()), candidates[0])
FEATURE_DIR = BASE / 'processed/training_features'
FINAL_DIR = BASE / 'models/gcan/final_model'
ABLATION_DIR = FINAL_DIR / 'missing_modality_experiments'
MANIFEST_PATH = BASE / 'manifests/final_manifest.csv'
CHECKPOINT_PATH = FINAL_DIR / 'gcan_final.pth'
RUN_DIR = Path(os.environ.get('NURTURENET_RUN_DIR', 'nurturenet_run')).resolve()
RUN_TRAINING = False
AUDIT_RAW_FILES = False  # Optional: scans the original dataset directories, without reading videos.
RUN_BENCHMARK = True
MODALITIES = ['video', 'audio', 'asr', 'ocr', 'metadata']
DIMS = [768, 768, 768, 768, 789]
CLASSES = ['Safe', 'Harmful Content', 'Adult Content', 'Suicide']
LABEL_TO_ID = dict(zip(CLASSES, range(4)))
ID_TO_LABEL = dict(enumerate(CLASSES))
EXPECTED_COUNTS = dict(zip(CLASSES, [1240, 1183, 1118, 1135]))
BATCH_SIZE = 64
MODEL_CONFIG = dict(hidden_dim=256, num_heads=8, num_classes=4, dropout=0.35, modality_dropout=0.10)
EPOCHS, LEARNING_RATE, WEIGHT_DECAY, PATIENCE = 40, 1e-4, 1e-4, 8
CLASS_WEIGHTS = [1.0, 1.15, 1.0, 1.0]
print('Project:', BASE)


Project: /Users/abdulrehman/Library/CloudStorage/GoogleDrive-abdulrehmantahir142@gmail.com/My Drive/NurtureNet


## 4. Dataset loading
Load the saved manifest and feature index, checking that video IDs, labels and splits agree.


In [3]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={'video_id': str})
index = pd.read_csv(FEATURE_DIR / 'feature_index.csv', dtype={'video_id': str})
assert manifest['video_id'].notna().all() and index['video_id'].notna().all()
assert manifest['video_id'].is_unique and index['video_id'].is_unique
assert set(manifest['video_id']) == set(index['video_id'])
joined = index[['video_id', 'label', 'split']].merge(
    manifest[['video_id', 'label', 'split']], on='video_id', validate='one_to_one', suffixes=('_feature', '_manifest'))
assert (joined['label_feature'] == joined['label_manifest']).all()
assert (joined['split_feature'] == joined['split_manifest']).all()
print('Manifest rows:', len(manifest), '| Feature index rows:', len(index))


Manifest rows: 4676 | Feature index rows: 4676


## 5. Cleaning and deduplication
Remove all records for 8 IDs with conflicting labels, then keep one record per remaining ID: **4,723 → 4,707 → 4,676 videos**. Optional raw-file auditing verifies this without deleting media.


In [4]:
def extract_video_id(filename):
    """
    Extract the TikTok video ID from filenames.
    Example:
    _englishvibes_7417006131060411653.mp4
    -> 7417006131060411653
    """
    match = re.search('(\\d{15,25})(?:\\.[^.]+)?$', filename)
    return match.group(1) if match else None

def build_manifest_3948(root):
    rows = []
    for split in ['train', 'val', 'test']:
        split_dir = root / split
        for label_dir in split_dir.iterdir():
            if not label_dir.is_dir():
                continue
            label = label_dir.name
            for video_path in label_dir.iterdir():
                if video_path.is_file() and video_path.suffix.lower() == '.mp4':
                    rows.append({'video_id': extract_video_id(video_path.name), 'filepath': str(video_path), 'dataset': 'Dataset_3948', 'split': split, 'label': label})
    return pd.DataFrame(rows)

def build_manifest_775(root):
    rows = []
    for label_dir in root.iterdir():
        if not label_dir.is_dir():
            continue
        label = label_dir.name
        for video_path in label_dir.iterdir():
            if video_path.is_file() and video_path.suffix.lower() == '.mp4':
                rows.append({'video_id': extract_video_id(video_path.name), 'filepath': str(video_path), 'dataset': 'Dataset_775', 'split': 'unsplit', 'label': label})
    return pd.DataFrame(rows)

if AUDIT_RAW_FILES:
    raw_3948 = build_manifest_3948(BASE / 'Dataset_3948/raw_dataset')
    raw_775 = build_manifest_775(BASE / 'Dataset_775/raw_dataset')
    assert (len(raw_3948), len(raw_775)) == (3948, 775)
    raw_manifest = pd.concat([raw_3948, raw_775], ignore_index=True)
    assert raw_manifest['video_id'].notna().all()
    label_counts = raw_manifest.groupby('video_id')['label'].nunique()
    conflict_ids = label_counts[label_counts > 1].index
    cleaned = raw_manifest[~raw_manifest['video_id'].isin(conflict_ids)].copy()
    unique = cleaned.drop_duplicates('video_id', keep='first').copy()
    assert len(conflict_ids) == 8 and len(cleaned) == 4707 and len(unique) == 4676
    assert set(unique['video_id']) == set(manifest['video_id'])
    assert unique.set_index('video_id')['label'].sort_index().equals(manifest.set_index('video_id')['label'].sort_index())
    print('Raw:', len(raw_manifest), '| Conflict IDs:', len(conflict_ids), '| Clean:', len(unique))
else:
    print('Using the preserved cleaned manifest; raw-file inventory is optional.')


Using the preserved cleaned manifest; raw-file inventory is optional.


**Archived output — `preprocessing.ipynb`, cell 9 (one-based).**

```text
Original rows: 4723
After removing conflicting IDs: 4707

Remaining duplicate video IDs:
Duplicate rows: 61
Unique duplicated IDs: 30
```


**Archived output — `preprocessing.ipynb`, cell 11 (one-based).**

```text
Rows before deduplication: 4707
Rows after deduplication : 4676
Videos removed           : 31

Unique video IDs:
4676

Label distribution:
label
Safe               1240
Harmful Content    1183
Suicide            1135
Adult Content      1118
Name: count, dtype: int64
```


## 6. Distributions and splits
Use the original stratified splits: **3,273 train / 701 validation / 702 test**, seed 42. Preserve saved membership and verify zero video-ID leakage.


In [5]:
def create_splits_from_ordered_manifest(ordered_unique):
    """For a new experiment only; preserve original row order to reproduce membership."""
    train, temporary = train_test_split(ordered_unique, test_size=0.30, stratify=ordered_unique['label'], random_state=SEED)
    val, test = train_test_split(temporary, test_size=0.50, stratify=temporary['label'], random_state=SEED)
    return pd.concat([frame.assign(split=name) for name, frame in [('train', train), ('val', val), ('test', test)]], ignore_index=True)

assert len(index) == 4676
assert index['label'].value_counts().to_dict() == EXPECTED_COUNTS
assert index['split'].value_counts().to_dict() == {'train': 3273, 'val': 701, 'test': 702}
assert index.groupby('video_id')['split'].nunique().max() == 1
if 'row_index' in index:
    assert np.array_equal(index['row_index'].to_numpy(), np.arange(len(index)))
print(pd.crosstab(index['split'], index['label'], margins=True).to_string())
print('Zero video-ID leakage verified.')


label  Adult Content  Harmful Content  Safe  Suicide   All
split                                                     
test             167              178   186      171   702
train            783              828   868      794  3273
val              168              177   186      170   701
All             1118             1183  1240     1135  4676
Zero video-ID leakage verified.


## 7. Feature loading / extraction
Load the original aligned feature arrays:

- **Video:** VideoMAE, 16 RGB frames and mean pooling → 768 dimensions.
- **Audio:** BEATs, 16-kHz mono audio → 768 dimensions.
- **ASR:** Whisper large-v3-turbo transcripts + mBERT → 768 dimensions.
- **OCR:** PP-OCRv5 text + mBERT → 768 dimensions.
- **Metadata:** 768 text + 21 structured dimensions → 789; imputation and scaling fit on training data only.

The saved arrays are reused to preserve the original experiment.


In [6]:
features = {name: np.load(FEATURE_DIR / f'{name}_features.npy', allow_pickle=False) for name in MODALITIES}
for name, dim in zip(MODALITIES, DIMS):
    array = features[name]
    assert array.shape == (len(index), dim), (name, array.shape)
    assert array.dtype == np.float32 and np.isfinite(array).all(), name
    print(f'{name:10}: {array.shape}, {array.dtype}, finite')
video, audio, asr, ocr, metadata = [features[name] for name in MODALITIES]


video     : (4676, 768), float32, finite
audio     : (4676, 768), float32, finite
asr       : (4676, 768), float32, finite
ocr       : (4676, 768), float32, finite
metadata  : (4676, 789), float32, finite


## 8. Modality availability masks
Saved binary masks follow `[video, audio, ASR, OCR, metadata]`. Each sample must retain at least one available modality.


In [7]:
masks = np.load(FEATURE_DIR / 'modality_masks.npy', allow_pickle=False)
assert masks.shape == (len(index), 5)
assert np.isfinite(masks).all() and set(np.unique(masks)) <= {0.0, 1.0}
assert (masks.sum(axis=1) > 0).all()
for i, name in enumerate(MODALITIES):
    print(f'{name:10}: {int(masks[:, i].sum())} available / {len(index)}')


video     : 4675 available / 4676
audio     : 4675 available / 4676
asr       : 4675 available / 4676
ocr       : 4208 available / 4676
metadata  : 4676 available / 4676


## 9. Dataset and DataLoader
Pair features, masks and labels in batches of 64. Shuffle training samples; preserve validation/test order.


In [8]:
labels = index['label'].map(LABEL_TO_ID).to_numpy(dtype=np.int64)
class NurtureNetDataset(Dataset):

    def __init__(self, row_indices):
        self.rows = np.asarray(row_indices, dtype=np.int64)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        i = self.rows[idx]
        return {'video': torch.from_numpy(video[i]), 'audio': torch.from_numpy(audio[i]), 'asr': torch.from_numpy(asr[i]), 'ocr': torch.from_numpy(ocr[i]), 'metadata': torch.from_numpy(metadata[i]), 'mask': torch.from_numpy(masks[i]), 'label': torch.tensor(labels[i], dtype=torch.long), 'row_index': torch.tensor(i, dtype=torch.long)}
train_idx, val_idx, test_idx = [np.flatnonzero(index['split'].to_numpy() == split) for split in ['train', 'val', 'test']]
train_dataset, val_dataset, test_dataset = [NurtureNetDataset(rows) for rows in [train_idx, val_idx, test_idx]]
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=device.type == 'cuda')
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=device.type == 'cuda')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=device.type == 'cuda')
print('Train / validation / test:', len(train_dataset), len(val_dataset), len(test_dataset))


Train / validation / test: 3273 701 702


## 10. GCAN V4
Project each modality to 256 dimensions, apply 8-head attention and a 256→512→256 FFN, then fuse available modalities using learned gates (256→128→1). The classifier maps 256→128→4. Dropout is 0.35, modality dropout 0.10; total parameters: **1,586,053**. Class logic comes from `inference.ipynb`, cell 2.


In [9]:
class GCAN_V4(nn.Module):

    def __init__(self, hidden_dim=256, num_heads=8, num_classes=4, dropout=0.35, modality_dropout=0.1):
        super().__init__()
        self.modality_dropout = modality_dropout
        self.video_proj = nn.Sequential(nn.Linear(768, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.audio_proj = nn.Sequential(nn.Linear(768, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.asr_proj = nn.Sequential(nn.Linear(768, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.ocr_proj = nn.Sequential(nn.Linear(768, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.metadata_proj = nn.Sequential(nn.Linear(789, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim * 2, hidden_dim), nn.Dropout(dropout))
        self.ffn_norm = nn.LayerNorm(hidden_dim)
        self.gate = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, num_classes))

    def apply_modality_dropout(self, mask):
        if not self.training or self.modality_dropout <= 0:
            return mask
        keep = (torch.rand_like(mask) > self.modality_dropout).float()
        new_mask = mask * keep
        empty = new_mask.sum(dim=1) == 0
        if empty.any():
            rows = torch.where(empty)[0]
            for row in rows:
                available = torch.where(mask[row] > 0)[0]
                chosen = available[torch.randint(len(available), (1,), device=mask.device)]
                new_mask[row, chosen] = 1.0
        return new_mask

    def forward(self, video, audio, asr, ocr, metadata, mask):
        mask = self.apply_modality_dropout(mask)
        v = self.video_proj(video)
        a = self.audio_proj(audio)
        s = self.asr_proj(asr)
        o = self.ocr_proj(ocr)
        m = self.metadata_proj(metadata)
        x = torch.stack([v, a, s, o, m], dim=1)
        mask_bool = mask.bool()
        x = x * mask.unsqueeze(-1)
        key_padding_mask = ~mask_bool
        (attn_out, attn_weights) = self.cross_attention(query=x, key=x, value=x, key_padding_mask=key_padding_mask, need_weights=True, average_attn_weights=False)
        x = self.attn_norm(x + attn_out)
        x = self.ffn_norm(x + self.ffn(x))
        x = x * mask.unsqueeze(-1)
        gate_logits = self.gate(x).squeeze(-1)
        gate_logits = gate_logits.masked_fill(~mask_bool, -1000000000.0)
        gate_weights = torch.softmax(gate_logits, dim=1)
        fused = torch.sum(x * gate_weights.unsqueeze(-1), dim=1)
        logits = self.classifier(fused)
        return {'logits': logits, 'gate_weights': gate_weights, 'attention_weights': attn_weights, 'fused': fused}

seed_everything()
model = GCAN_V4(**MODEL_CONFIG).to(device)
parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count == 1_586_053
print('GCAN V4 parameters:', f'{parameter_count:,}')


GCAN V4 parameters: 1,586,053


## 11. Training configuration
Use AdamW (LR/weight decay 1e-4), weighted cross-entropy `[1, 1.15, 1, 1]` and gradient clipping 1.0. Train up to 40 epochs, stop after 8 without validation-F1 improvement, and reduce LR by 0.5 after 3 plateau epochs (minimum 1e-6).


In [10]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS, device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
print(dict(optimizer='AdamW', learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
           batch_size=BATCH_SIZE, max_epochs=EPOCHS, early_stop=PATIENCE,
           grad_clip=1.0, class_weights=CLASS_WEIGHTS))


{'optimizer': 'AdamW', 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'batch_size': 64, 'max_epochs': 40, 'early_stop': 8, 'grad_clip': 1.0, 'class_weights': [1.0, 1.15, 1.0, 1.0]}


## 12. Training and validation
The reconstructed loop applies the saved V4 configuration to the source training code. Default mode displays archived history; enable `RUN_TRAINING` for a new run. Loss retains the source’s batch-size-weighted averaging convention.


In [11]:
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    (precision, recall, f1, _) = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

def move_batch_to_device(batch):
    return {k: v.to(device, non_blocking=True) for (k, v) in batch.items() if k != 'row_index'}

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    all_true = []
    all_pred = []
    for batch in loader:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        output = model(video=batch['video'], audio=batch['audio'], asr=batch['asr'], ocr=batch['ocr'], metadata=batch['metadata'], mask=batch['mask'])
        logits = output['logits']
        loss = criterion(logits, batch['label'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * batch['label'].size(0)
        predictions = logits.argmax(dim=1)
        all_true.extend(batch['label'].detach().cpu().numpy())
        all_pred.extend(predictions.detach().cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_true, all_pred)
    return (epoch_loss, metrics)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    all_true = []
    all_pred = []
    for batch in loader:
        batch = move_batch_to_device(batch)
        output = model(video=batch['video'], audio=batch['audio'], asr=batch['asr'], ocr=batch['ocr'], metadata=batch['metadata'], mask=batch['mask'])
        logits = output['logits']
        loss = criterion(logits, batch['label'])
        running_loss += loss.item() * batch['label'].size(0)
        predictions = logits.argmax(dim=1)
        all_true.extend(batch['label'].cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_true, all_pred)
    return (epoch_loss, metrics, np.asarray(all_true), np.asarray(all_pred))


In [12]:
history = []
if RUN_TRAINING:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    seed_everything()
    best_val_f1, stale = -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_metrics = train_one_epoch(model, train_loader)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader)
        scheduler.step(val_metrics['f1'])
        history.append(dict(epoch=epoch, train_loss=train_loss, train_accuracy=train_metrics['accuracy'],
                            train_f1=train_metrics['f1'], val_loss=val_loss, val_accuracy=val_metrics['accuracy'],
                            val_precision=val_metrics['precision'], val_recall=val_metrics['recall'],
                            val_f1=val_metrics['f1'], learning_rate=optimizer.param_groups[0]['lr']))
        if val_metrics['f1'] > best_val_f1:
            best_val_f1, stale = val_metrics['f1'], 0
            torch.save(dict(epoch=epoch, model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                            val_f1=best_val_f1, label_to_id=LABEL_TO_ID, class_weights=torch.tensor(CLASS_WEIGHTS),
                            model_config={**MODEL_CONFIG, 'version': 'GCAN_V4'}), RUN_DIR / 'gcan_v4_reconstructed_best.pth')
        else:
            stale += 1
        print(f"Epoch {epoch:02d} | train F1 {train_metrics['f1']:.4f} | validation F1 {val_metrics['f1']:.4f}")
        if stale >= PATIENCE:
            break
    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_DIR / 'training_history.csv', index=False)
else:
    history_df = pd.read_csv(FINAL_DIR / 'training_history.csv')
    print('Archived V4 training history; no retraining performed.')
print(history_df.to_string(index=False))


Archived V4 training history; no retraining performed.
 epoch  train_loss  train_accuracy  train_f1  val_loss  val_accuracy  val_precision  val_recall   val_f1  learning_rate
     1    1.158256        0.491292  0.489765  0.851027      0.691869       0.700067    0.692133 0.692831       0.000100
     2    0.851768        0.684082  0.683683  0.694872      0.750357       0.750268    0.752775 0.749515       0.000100
     3    0.723492        0.746410  0.745793  0.641309      0.778887       0.779827    0.780579 0.779196       0.000100
     4    0.639148        0.782768  0.782684  0.601624      0.798859       0.800405    0.799912 0.797238       0.000100
     5    0.586591        0.792851  0.792593  0.591429      0.810271       0.817486    0.810604 0.807768       0.000100
     6    0.529343        0.816071  0.816118  0.577215      0.805991       0.806820    0.806867 0.805030       0.000100
     7    0.485300        0.832875  0.833029  0.582066      0.815977       0.822156    0.817484 0.814847 

## 13. Model selection and checkpoint loading
Select by validation Macro-F1. Load historical epoch 12 by default, or the new run’s best checkpoint when training is enabled.


In [13]:
selected_path = RUN_DIR / 'gcan_v4_reconstructed_best.pth' if RUN_TRAINING else CHECKPOINT_PATH
checkpoint = torch.load(selected_path, map_location=device, weights_only=False)
assert checkpoint['label_to_id'] == LABEL_TO_ID
for key, value in MODEL_CONFIG.items():
    assert checkpoint['model_config'][key] == value
assert torch.allclose(checkpoint['class_weights'].cpu(), torch.tensor(CLASS_WEIGHTS))
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()
summary = json.loads((FINAL_DIR / 'final_model_summary.json').read_text())
assert summary['selected_model'] == 'GCAN_V4'
if not RUN_TRAINING:
    assert checkpoint['epoch'] == summary['training']['best_epoch'] == 12
    assert abs(checkpoint['val_f1'] - history_df['val_f1'].max()) < 1e-10
print('Checkpoint epoch:', checkpoint['epoch'], '| Validation Macro-F1:', checkpoint['val_f1'])


Checkpoint epoch: 12 | Validation Macro-F1: 0.8165259921310235


## 14. Final test evaluation
Compute loss, accuracy and macro precision/recall/F1 on the held-out test set; compare against the saved V4 results.


In [14]:
test_loss, test_metrics, y_true, y_pred = evaluate(model, test_loader)
print(f'Test loss: {test_loss:.4f}')
for name, value in test_metrics.items():
    print(f'{name}: {100 * value:.2f}%')
if not RUN_TRAINING:
    for key, saved_key in [('accuracy', 'accuracy'), ('precision', 'macro_precision'), ('recall', 'macro_recall'), ('f1', 'macro_f1')]:
        assert np.isclose(test_metrics[key], summary['test'][saved_key], atol=1e-10), key
    assert np.isclose(test_loss, summary['test']['loss'], atol=1e-5)


Test loss: 0.6340
accuracy: 80.06%
precision: 80.31%
recall: 80.07%
f1: 79.93%


## 15. Controlled video-only baseline
The separately trained **VideoMAE + MLP** baseline achieved **71.51% accuracy / 71.53% Macro-F1**. Load its archived history and reports; the original implementation is missing, so these results are not claimed as rerun.


In [15]:
baseline_comparison = pd.read_csv(ABLATION_DIR / 'video_only_vs_gcan.csv')
baseline_report = pd.read_csv(ABLATION_DIR / 'video_only_classification_report.csv', index_col=0)
baseline_history = pd.read_csv(ABLATION_DIR / 'video_only_training_history.csv')
print('Archived controlled baseline comparison:')
print(baseline_comparison.to_string(index=False))
baseline_row = baseline_comparison.iloc[0]
assert np.isclose(baseline_row['accuracy'], baseline_report.loc['accuracy', 'precision'])
assert np.isclose(baseline_row['macro_f1'], baseline_report.loc['macro avg', 'f1-score'])
print('Baseline best validation epoch:', int(baseline_history.loc[baseline_history['val_macro_f1'].idxmax(), 'epoch']))
print(f"Accuracy gain: {(test_metrics['accuracy'] - baseline_row['accuracy']) * 100:.2f} pp")
print(f"Macro-F1 gain: {(test_metrics['f1'] - baseline_row['macro_f1']) * 100:.2f} pp")


Archived controlled baseline comparison:
                  model  accuracy  macro_precision  macro_recall  macro_f1
VideoMAE features + MLP   0.71510         0.719434      0.716986  0.715296
        NurtureNet GCAN   0.80057         0.803111      0.800745  0.799265
Baseline best validation epoch: 30
Accuracy gain: 8.55 pp
Macro-F1 gain: 8.40 pp


## 16. Missing-modality ablations
Remove one modality through its mask without retraining. Report Macro-F1 change versus the full model; reject samples left with no available modality.


In [16]:
@torch.no_grad()
def evaluate_ablation(removed):
    model.eval()
    true, predicted = [], []
    for batch in test_loader:
        batch = move_batch_to_device(batch)
        mask = batch['mask'].clone()
        mask[:, MODALITIES.index(removed)] = 0
        assert (mask.sum(dim=1) > 0).all(), 'Ablation leaves an all-missing sample; policy must be resolved.'
        inputs = {name: batch[name] for name in MODALITIES}
        output = model(**inputs, mask=mask)
        true.extend(batch['label'].cpu().numpy())
        predicted.extend(output['logits'].argmax(1).cpu().numpy())
    return calculate_metrics(true, predicted)

ablation_rows = []
for name in MODALITIES:
    result = evaluate_ablation(name)
    ablation_rows.append(dict(modality=name, **result, delta_f1_pp=100 * (result['f1'] - test_metrics['f1'])))
ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
archived_ablation = pd.read_csv(ABLATION_DIR / 'single_modality_removal_results.csv')
if not RUN_TRAINING:
    for row in ablation_rows:
        expected = archived_ablation.loc[archived_ablation['removed_modality'] == row['modality'], 'macro_f1'].iloc[0]
        assert np.isclose(row['f1'], expected, atol=1e-10)


modality  accuracy  precision  recall     f1  delta_f1_pp
   video    0.7493     0.7482  0.7488 0.7470      -5.2291
   audio    0.7892     0.7979  0.7894 0.7876      -1.1680
     asr    0.8034     0.8042  0.8035 0.8029       0.3594
     ocr    0.7906     0.7945  0.7913 0.7907      -0.8608
metadata    0.7707     0.7706  0.7715 0.7704      -2.8889


## 17. Per-class performance
Report precision, recall, F1 and support for each of the four classes.


In [17]:
report = classification_report(y_true, y_pred, labels=range(4), target_names=CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
print(report_df.to_string(float_format=lambda value: f'{value:.4f}'))
archived_report = pd.read_csv(FINAL_DIR / 'test_classification_report.csv', index_col=0)
if not RUN_TRAINING:
    assert np.allclose(report_df.loc[CLASSES, ['precision', 'recall', 'f1-score']], archived_report.loc[CLASSES, ['precision', 'recall', 'f1-score']], atol=1e-10)


                 precision  recall  f1-score  support
Safe                0.7585  0.8441    0.7990 186.0000
Harmful Content     0.8276  0.6742    0.7430 178.0000
Adult Content       0.7500  0.7725    0.7611 167.0000
Suicide             0.8764  0.9123    0.8940 171.0000
accuracy            0.8006  0.8006    0.8006   0.8006
macro avg           0.8031  0.8007    0.7993 702.0000
weighted avg        0.8027  0.8006    0.7989 702.0000


## 18. Confusion matrix
Rows show true classes; columns show predictions. Compare the computed matrix with the saved V4 matrix.


In [18]:
cm = confusion_matrix(y_true, y_pred, labels=range(4))
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
print(cm_df.to_string())
if not RUN_TRAINING:
    archived_cm = pd.read_csv(FINAL_DIR / 'test_confusion_matrix.csv', index_col=0)
    assert np.array_equal(cm, archived_cm.to_numpy())


                 Safe  Harmful Content  Adult Content  Suicide
Safe              157                8             15        6
Harmful Content    27              120             22        9
Adult Content      17               14            129        7
Suicide             6                3              6      156


## 19. Qualitative examples
Re-evaluate four archived test examples and display probabilities, masks and gate weights. Gates indicate model allocation, not causal explanations.


In [19]:
qualitative_archive = pd.read_csv(FINAL_DIR / 'final_v4_qualitative_examples.csv', dtype={'video_id': str})
qualitative_rows = []
model.eval()
with torch.no_grad():
    for video_id in qualitative_archive['video_id']:
        matches = np.flatnonzero(index['video_id'].to_numpy() == video_id)
        assert len(matches) == 1 and index.iloc[matches[0]]['split'] == 'test'
        i = int(matches[0])
        inputs = {name: torch.from_numpy(features[name][i:i+1]).to(device) for name in MODALITIES}
        output = model(**inputs, mask=torch.from_numpy(masks[i:i+1]).to(device))
        probabilities = output['logits'].softmax(1)[0].cpu().numpy()
        gates = output['gate_weights'][0].cpu().numpy()
        record = dict(video_id=video_id, true_label=index.iloc[i]['label'], predicted_label=CLASSES[int(probabilities.argmax())], confidence=float(probabilities.max()))
        record.update(dict(zip(['p_safe', 'p_harmful', 'p_adult', 'p_suicide'], probabilities.tolist())))
        record.update({f'mask_{name}': int(masks[i, j]) for j, name in enumerate(MODALITIES)})
        record.update({f'gate_{name}': float(gates[j]) for j, name in enumerate(MODALITIES)})
        qualitative_rows.append(record)
qualitative_df = pd.DataFrame(qualitative_rows)
print(qualitative_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
if not RUN_TRAINING:
    assert qualitative_df['predicted_label'].tolist() == qualitative_archive['predicted_label'].tolist()
    assert np.allclose(qualitative_df[['p_safe','p_harmful','p_adult','p_suicide']], qualitative_archive[['p_safe','p_harmful','p_adult','p_suicide']], atol=1e-5)


           video_id      true_label predicted_label  confidence  p_safe  p_harmful  p_adult  p_suicide  mask_video  mask_audio  mask_asr  mask_ocr  mask_metadata  gate_video  gate_audio  gate_asr  gate_ocr  gate_metadata
7282361849658592513            Safe            Safe      0.9758  0.9758     0.0104   0.0094     0.0044           1           1         1         1              1      0.2287      0.2278    0.2276    0.1194         0.1966
6928836923398049030 Harmful Content Harmful Content      0.9874  0.0058     0.9874   0.0036     0.0032           1           1         1         0              1      0.2412      0.3504    0.0919    0.0000         0.3165
7221906826533539098   Adult Content   Adult Content      0.9838  0.0063     0.0078   0.9838     0.0021           1           1         1         1              1      0.2223      0.3434    0.1172    0.1296         0.1875
7062539048228949295         Suicide         Suicide      0.9913  0.0026     0.0038   0.0023     0.9913           1  

## 20. Computational benchmark
Measure current GCAN forward-pass latency on precomputed features. This excludes extraction and transfer time and is not an archived end-to-end benchmark.


In [20]:
if RUN_BENCHMARK:
    benchmark_batch = move_batch_to_device(next(iter(test_loader)))
    inputs = {name: benchmark_batch[name] for name in MODALITIES}
    inputs['mask'] = benchmark_batch['mask']
    def synchronize():
        if device.type == 'cuda':
            torch.cuda.synchronize()
    model.eval()
    with torch.no_grad():
        for _ in range(10):
            model(**inputs)
        synchronize()
        timings = []
        for _ in range(50):
            start = time.perf_counter()
            model(**inputs)
            synchronize()
            timings.append(time.perf_counter() - start)
    batch_ms = 1000 * float(np.median(timings))
    print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
    print('Threads:', torch.get_num_threads(), '| Batch:', len(benchmark_batch['label']))
    print(f'Median batch latency: {batch_ms:.3f} ms | per sample: {batch_ms / len(benchmark_batch["label"]):.3f} ms')
    print(f'Throughput: {len(benchmark_batch["label"]) / np.median(timings):.1f} samples/s')


Device: CPU
Threads: 4 | Batch: 64
Median batch latency: 1.519 ms | per sample: 0.024 ms
Throughput: 42120.9 samples/s


## 21. Final summary and consistency audit
**Verified:** dataset counts/splits, V4 architecture, test metrics, per-class scores and ablations match the requested values. The earlier training notebook’s 79.34% accuracy / 79.37% F1 belongs to an obsolete run.

**Source limitations and repairs:**
- Original V4 training and feature/mask assembly code are missing; training is reconstructed, and original arrays are required.
- The baseline notebook is empty; its results are supported by archived reports.
- The missing conflict-ID definition was repaired using label disagreement. Saved splits avoid dependence on unknown original file order.
- BEATs processes full audio without explicit speech removal; complete OCR feature-version lineage is unavailable.
- Metadata preprocessing files are under `processed/preprocessing_config`; inference used stale paths.
- No historical end-to-end benchmark was found.

The scheduler uses its default relative threshold (1e-4), while checkpoint selection accepts any F1 increase; this explains the epoch-13 LR reduction after the epoch-12 checkpoint.


In [21]:
audit_targets = {'loss': 0.6340, 'accuracy': 80.06, 'precision': 80.31, 'recall': 80.07, 'f1': 79.93}
print('Current evaluation vs paper targets:')
print(f"loss: {test_loss:.4f} (target {audit_targets['loss']:.4f})")
for name in ['accuracy', 'precision', 'recall', 'f1']:
    value = round(100 * test_metrics[name], 2)
    print(f"{name}: {value:.2f}% | target {audit_targets[name]:.2f}% | match={value == audit_targets[name]}")
expected_class_f1 = [79.90, 74.30, 76.11, 89.40]
for name, target in zip(CLASSES, expected_class_f1):
    value = round(100 * report[name]['f1-score'], 2)
    print(f'{name} F1: {value:.2f}% | target {target:.2f}% | match={value == target}')
expected_ablation = dict(video=-5.23, metadata=-2.89, audio=-1.17, ocr=-0.86, asr=0.36)
for row in ablation_rows:
    value = round(row['delta_f1_pp'], 2)
    print(f"No {row['modality']}: {value:+.2f} pp | target {expected_ablation[row['modality']]:+.2f} pp")
print('Fresh-training mode:', RUN_TRAINING)


Current evaluation vs paper targets:
loss: 0.6340 (target 0.6340)
accuracy: 80.06% | target 80.06% | match=True
precision: 80.31% | target 80.31% | match=True
recall: 80.07% | target 80.07% | match=True
f1: 79.93% | target 79.93% | match=True
Safe F1: 79.90% | target 79.90% | match=True
Harmful Content F1: 74.30% | target 74.30% | match=True
Adult Content F1: 76.11% | target 76.11% | match=True
Suicide F1: 89.40% | target 89.40% | match=True
No video: -5.23 pp | target -5.23 pp
No audio: -1.17 pp | target -1.17 pp
No asr: +0.36 pp | target +0.36 pp
No ocr: -0.86 pp | target -0.86 pp
No metadata: -2.89 pp | target -2.89 pp
Fresh-training mode: False
